In [1]:
import pandas as pd
import data_analysis_utils as utils
import lda_class as lda_class 
from sklearn.preprocessing import StandardScaler


In [2]:

inp_dataset = pd.read_csv('wine.csv')
inp_dataset


,Alcohol,Malic_Acid,Ash,Ash_Alcanity,Magnesium,Total_Phenols,Flavanoids,Nonflavanoid_Phenols,Proanthocyanins,Color_Intensity,Hue,OD280,Proline,Customer_Segment
0,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065,1
1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050,1
2,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185,1
3,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480,1
4,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740,3
174,13.40,3.91,2.48,23.0,102,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750,3
175,13.27,4.28,2.26,20.0,120,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835,3
176,13.17,2.59,2.37,20.0,120,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840,3


In [3]:
print('Missing Value Percentages:', inp_dataset.isnull().mean()*100)

quan, qual = utils.Preprocessing.quanQual(inp_dataset)
inp_dataset = utils.Preprocessing.quanVariables(inp_dataset, qual)


Missing Value Percentages: Alcohol                 0.0
Malic_Acid              0.0
Ash                     0.0
Ash_Alcanity            0.0
Magnesium               0.0
Total_Phenols           0.0
Flavanoids              0.0
Nonflavanoid_Phenols    0.0
Proanthocyanins         0.0
Color_Intensity         0.0
Hue                     0.0
OD280                   0.0
Proline                 0.0
Customer_Segment        0.0
dtype: float64


In [4]:
indep = inp_dataset.iloc[:, :-1]
dep = inp_dataset.iloc[:, -1]
if len(qual) > 0:
    quan_data = indep[quan]
else:
    quan_data = indep

quan_scaler = StandardScaler()
quan_scaled = quan_scaler.fit_transform(quan_data)

if len(qual) > 0:
    scaled_data = pd.DataFrame(quan_scaled, columns=quan)
    scaled_data[qual] = inp_dataset[qual]
    data = pd.get_dummies(scaled_data, drop_first=True)
else:
    data = quan_scaled

indep_X = data
dep_Y = dep


In [5]:
n_classes = len(dep_Y.unique())
max_components = min(indep_X.shape[1], n_classes - 1)
print("Max allowable LDA components:", max_components)

fullresult = pd.DataFrame()

for comp_no in range(1, max_components+1):
    result = lda_class.lda_classifiers(indep_X, dep_Y, comp_no)
    fullresult = pd.concat([fullresult, result], ignore_index=True)


fullresult


Max allowable LDA components: 2


,No_Of_Components,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
0,1,0.844444,0.844444,0.888889,0.844444,0.844444,0.822222,0.844444
1,2,0.955556,0.955556,0.911111,0.911111,0.933333,0.933333,0.955556


In [6]:
max_val = fullresult.iloc[:, 1:].values.max()
row, col = fullresult.iloc[:, 1:].stack().idxmax()
no_features = fullresult.iloc[row, 0]
print(f'Max Accuracy is: {max_val:.4f}% for {col} Model  \nWith No of Features = {no_features}')


Max Accuracy is: 0.9556% for Logistic Model  
With No of Features = 2
